In [ ]:
# import modules
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import optimizers
from datetime import datetime
from kerastuner.tuners import RandomSearch
import tensorflow_addons as tfa
from keras.utils import to_categorical 
from keras.regularizers import l2
import os



r_seed = 42

from pathlib import Path
WORKDIR = str(Path.cwd().parent) + '/'  # raiz do repositório
np.random.seed(r_seed)
tf.random.set_seed(r_seed)
data_hora= datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
LOG_DIR = WORKDIR + "/logs/"+data_hora

In [ ]:
pd.set_option("display.precision", 9)

In [ ]:
date = datetime.now()
date = date.strftime("%d/%m/%Y_%H:%M")
arq = datetime.now()
arq = arq.strftime("%d_%m_%Y_%H_%M")

In [ ]:
def equalize_dataframe_sizes(df1, df2):
    """Equalizes the sizes of two DataFrames by dropping rows from the larger one.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        tuple: A tuple containing the equalized DataFrames (df1, df2).

    Raises:
        ValueError: If both DataFrames have the same size.
    """

    if df1.shape[0] == df2.shape[0]:
        raise ValueError("Both DataFrames already have the same size.")

    larger_df, smaller_df = (df1, df2) if df1.shape[0] > df2.shape[0] else (df2, df1)

    # Drop rows from the larger DataFrame to match the smaller one
    larger_df = larger_df.iloc[: smaller_df.shape[0], :]

    return larger_df, smaller_df

# Example usage
#df1 = pd.DataFrame({'A': [1, 2, 3, 4, 5], 'B': ['a', 'b', 'c', 'd', 'e']})
#df2 = pd.DataFrame({'A': [10, 20, 30], 'B': ['x', 'y', 'z']})

''' try:
    df1_equal, df2_equal = equalize_dataframe_sizes(df1.copy(), df2.copy())
    #print("Equalized DataFrames:")
    #print(df1_equal)
    #print(df2_equal)
except ValueError as e:
    print(e) '''


In [107]:
def custom_concat(df1, df2):
    """
    Concatenates two DataFrames with the same column names, considering specific conditions.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        pd.DataFrame: The concatenated DataFrame.
    """

    result_df = pd.DataFrame()

    # Create groups based on unique combinations of X and Y values
    groups_df1 = df1.groupby(['pessoa_x_watch','atividade_watch'])
    groups_df2 = df2.groupby(['pessoa', 'ativ'])

    # Iterate over groups from both DataFrames simultaneously
    for (x, y), group1 in groups_df1:       
        group2 = groups_df2.get_group((x, y))     
        if group2 is not None:  # Check if corresponding group exists in df2
            if group1.shape[0] != group2.shape[0]:
                group1, group2 = equalize_dataframe_sizes(group1.copy(), group2.copy())
                
            group1 = group1.reset_index(drop=True) 
            group2 = group2.reset_index(drop=True)  
            df_concat_group1_group2 = pd.concat([group1, group2], axis=1)
            #print(df_concat_group1_group2)
            df_concat_group1_group2 = df_concat_group1_group2.dropna()            
            #print(df_concat_group1_group2)
            has_nan = df_concat_group1_group2.isnull().sum().sum() > 0
            if has_nan == True:
                print (f"**************************** Imprimindo concat co nan: {df_concat_group1_group2}")
                break
            df_concat_group1_group2 = df_concat_group1_group2.reset_index(drop=True) 
            result_df = result_df.reset_index(drop=True)
            result_df = pd.concat([result_df, df_concat_group1_group2])

    return result_df

In [108]:
df_watch = pd.read_csv("activity_recognition_data_watch_concat_by_comparacao_grupos.csv",dtype={"value":"float64"})

In [109]:
df_watch.head()

,pessoa_x,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,0,90426708196641,7.0916250,-0.59166710,8.1955020,90426757696641,0.314944100,-1.02227650,-0.309961600
1,1600,0,90426757696641,4.9727570,-0.15831658,6.6967316,90426807196641,0.387382100,-0.61854124,-0.048971802
2,1600,0,90426807196641,3.2537200,-0.19183542,6.1077580,90426856696641,0.070998540,-0.20947972,-0.195978300
3,1600,0,90426856696641,2.8012160,-0.15592238,5.9976250,90426906196641,0.037975345,0.25497556,-0.156563500
4,1600,0,90426906196641,3.7708676,-1.05135380,7.7310270,90426955696641,0.073129070,0.71943086,-0.001034904


In [110]:
df_watch.shape

(3441622, 10)

In [111]:
df_watch.atividade.value_counts().sort_index()

atividade
0     188925
1     184220
2     187622
3     191446
4     186899
5     190777
6     187144
7     194261
8     189816
9     193210
10    194315
11    197393
12    187927
13    187672
14    191241
15    201004
16    187171
17    200579
Name: count, dtype: int64

In [112]:
df_watch['atividade_watch'] = df_watch['atividade']

In [113]:
df_watch['pessoa_x_watch'] = df_watch['pessoa_x']

In [114]:
df_watch = df_watch.drop('pessoa_x',axis=1)

In [115]:
df_watch = df_watch.drop('atividade',axis=1)

In [116]:
df_watch.head()

,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z,atividade_watch,pessoa_x_watch
0,90426708196641,7.0916250,-0.59166710,8.1955020,90426757696641,0.314944100,-1.02227650,-0.309961600,0,1600
1,90426757696641,4.9727570,-0.15831658,6.6967316,90426807196641,0.387382100,-0.61854124,-0.048971802,0,1600
2,90426807196641,3.2537200,-0.19183542,6.1077580,90426856696641,0.070998540,-0.20947972,-0.195978300,0,1600
3,90426856696641,2.8012160,-0.15592238,5.9976250,90426906196641,0.037975345,0.25497556,-0.156563500,0,1600
4,90426906196641,3.7708676,-1.05135380,7.7310270,90426955696641,0.073129070,0.71943086,-0.001034904,0,1600


In [117]:
df_watch = df_watch.loc[~((df_watch['pessoa_x_watch'] == 1607) & (df_watch['atividade_watch'] == 9))]


In [118]:
#df_watch = df_watch[(df_watch['pessoa_x_watch'] != 1607) & (df_watch['atividade_watch'] != 9)]

In [119]:
df_watch.atividade_watch.value_counts().sort_index()

atividade_watch
0     188925
1     184220
2     187622
3     191446
4     186899
5     190777
6     187144
7     194261
8     189816
9     189608
10    194315
11    197393
12    187927
13    187672
14    191241
15    201004
16    187171
17    200579
Name: count, dtype: int64

In [120]:
df_watch = df_watch.loc[~((df_watch['pessoa_x_watch'] == 1609) & (df_watch['atividade_watch'] == 1))]

In [121]:
#df_watch = df_watch[(df_watch['pessoa_x_watch']!=1609) & (df_watch['atividade_watch']!=1) ]

In [122]:
df_watch = df_watch.loc[~((df_watch['pessoa_x_watch'] == 1616) & (df_watch['atividade_watch'] == 5))]

In [123]:
#df_watch = df_watch[(df_watch['pessoa_x_watch']!=1616) & (df_watch['atividade_watch']!=5) ]

In [124]:
df_watch = df_watch.loc[~((df_watch['pessoa_x_watch'] == 1641) & (df_watch['atividade_watch'] == 15))]

In [125]:
#df_watch = df_watch[(df_watch['pessoa_x_watch']!=1641) & (df_watch['atividade_watch']!=15) ]

In [126]:
df_watch = df_watch.loc[~((df_watch['pessoa_x_watch'] == 1644) & (df_watch['atividade_watch'] == 11))]

In [127]:
#df_watch = df_watch[(df_watch['pessoa_x_watch']!=1644) & (df_watch['atividade_watch']!=11) ]

In [128]:
df_watch = df_watch.loc[~((df_watch['pessoa_x_watch'] == 1643) & (df_watch['atividade_watch'] == 8))]

In [129]:
#df_watch = df_watch[(df_watch['pessoa_x_watch']!=1643) & (df_watch['atividade_watch']!=8) ]

In [130]:
df_watch.head()

,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z,atividade_watch,pessoa_x_watch
0,90426708196641,7.0916250,-0.59166710,8.1955020,90426757696641,0.314944100,-1.02227650,-0.309961600,0,1600
1,90426757696641,4.9727570,-0.15831658,6.6967316,90426807196641,0.387382100,-0.61854124,-0.048971802,0,1600
2,90426807196641,3.2537200,-0.19183542,6.1077580,90426856696641,0.070998540,-0.20947972,-0.195978300,0,1600
3,90426856696641,2.8012160,-0.15592238,5.9976250,90426906196641,0.037975345,0.25497556,-0.156563500,0,1600
4,90426906196641,3.7708676,-1.05135380,7.7310270,90426955696641,0.073129070,0.71943086,-0.001034904,0,1600


In [131]:
df_watch.shape

(3420018, 10)

In [132]:
df_watch.atividade_watch.value_counts().sort_index()

atividade_watch
0     188925
1     180617
2     187622
3     191446
4     186899
5     187174
6     187144
7     194261
8     186218
9     189608
10    194315
11    193794
12    187927
13    187672
14    191241
15    197405
16    187171
17    200579
Name: count, dtype: int64

In [133]:
df_watch.columns

Index(['tempo_acelerômetro', 'acelerômetro_x', 'acelerômetro_Y',
       'acelerômetro_Z', 'tempo_gyro', 'giroscópio_x', 'giroscópio_Y',
       'giroscópio_Z', 'atividade_watch', 'pessoa_x_watch'],
      dtype='object')

In [134]:
df_watch = df_watch [['pessoa_x_watch','atividade_watch', 'tempo_acelerômetro', 'acelerômetro_x', 'acelerômetro_Y',
       'acelerômetro_Z', 'tempo_gyro', 'giroscópio_x', 'giroscópio_Y',
       'giroscópio_Z']]

In [135]:
df_watch.head()

,pessoa_x_watch,atividade_watch,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,0,90426708196641,7.0916250,-0.59166710,8.1955020,90426757696641,0.314944100,-1.02227650,-0.309961600
1,1600,0,90426757696641,4.9727570,-0.15831658,6.6967316,90426807196641,0.387382100,-0.61854124,-0.048971802
2,1600,0,90426807196641,3.2537200,-0.19183542,6.1077580,90426856696641,0.070998540,-0.20947972,-0.195978300
3,1600,0,90426856696641,2.8012160,-0.15592238,5.9976250,90426906196641,0.037975345,0.25497556,-0.156563500
4,1600,0,90426906196641,3.7708676,-1.05135380,7.7310270,90426955696641,0.073129070,0.71943086,-0.001034904


In [136]:
df_watch.atividade_watch.value_counts()

atividade_watch
17    200579
15    197405
10    194315
7     194261
11    193794
3     191446
14    191241
9     189608
0     188925
12    187927
13    187672
2     187622
5     187174
16    187171
6     187144
4     186899
8     186218
1     180617
Name: count, dtype: int64

In [137]:
df_watch.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3420018 entries, 0 to 3441621
Data columns (total 10 columns):
 #   Column              Dtype  
---  ------              -----  
 0   pessoa_x_watch      int64  
 1   atividade_watch     int64  
 2   tempo_acelerômetro  int64  
 3   acelerômetro_x      float64
 4   acelerômetro_Y      float64
 5   acelerômetro_Z      float64
 6   tempo_gyro          int64  
 7   giroscópio_x        float64
 8   giroscópio_Y        float64
 9   giroscópio_Z        float64
dtypes: float64(6), int64(4)
memory usage: 287.0 MB


In [138]:
df_phone = pd.read_csv("activity_recognition_data_phone_concat_by_comparacao_grupos.csv",dtype={"value":"float64"})

In [139]:
df_phone.shape

(3608408, 10)

In [140]:
df_phone.head()

,pessoa_x,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,0,252207666810782,-0.36476135,8.793503,1.05508420,252207918580802,-0.85321045,0.297225950,0.89018250
1,1600,0,252207717164786,-0.87973020,9.768784,1.01699830,252207968934806,-0.87513730,0.015472412,0.16223145
2,1600,0,252207767518790,2.00149540,11.109070,2.61915600,252208019288809,-0.72016907,0.388488770,-0.28401184
3,1600,0,252207817872794,0.45062256,12.651642,0.18455505,252208069642813,-0.57164000,1.227401700,-0.24166870
4,1600,0,252207868226798,-2.16435240,13.928436,-4.42248540,252208119996817,-0.38049316,1.202835100,-0.21313477


In [141]:
df_phone['atividade_phone'] = df_phone['atividade']

In [142]:
df_phone['pessoa_x_phone'] = df_phone['pessoa_x']

In [143]:
df_phone = df_phone.drop('pessoa_x',axis=1)

In [144]:
df_phone = df_phone.drop('atividade',axis=1)

In [145]:
df_phone.columns

Index(['tempo_acelerômetro', 'acelerômetro_x', 'acelerômetro_Y',
       'acelerômetro_Z', 'tempo_gyro', 'giroscópio_x', 'giroscópio_Y',
       'giroscópio_Z', 'atividade_phone', 'pessoa_x_phone'],
      dtype='object')

In [146]:
df_phone = df_phone [['pessoa_x_phone','atividade_phone','tempo_acelerômetro', 'acelerômetro_x', 'acelerômetro_Y',
       'acelerômetro_Z', 'tempo_gyro', 'giroscópio_x', 'giroscópio_Y',
       'giroscópio_Z']]

In [147]:
df_phone.columns= ['pessoa','ativ','tempo_acel', 'acel_x_', 'acel_Y',
       'acel_Z', 'temp_gyro', 'giro_x', 'giro_Y_',
       'giro_Z']

In [148]:
df_phone.head()

,pessoa,ativ,tempo_acel,acel_x_,acel_Y,acel_Z,temp_gyro,giro_x,giro_Y_,giro_Z
0,1600,0,252207666810782,-0.36476135,8.793503,1.05508420,252207918580802,-0.85321045,0.297225950,0.89018250
1,1600,0,252207717164786,-0.87973020,9.768784,1.01699830,252207968934806,-0.87513730,0.015472412,0.16223145
2,1600,0,252207767518790,2.00149540,11.109070,2.61915600,252208019288809,-0.72016907,0.388488770,-0.28401184
3,1600,0,252207817872794,0.45062256,12.651642,0.18455505,252208069642813,-0.57164000,1.227401700,-0.24166870
4,1600,0,252207868226798,-2.16435240,13.928436,-4.42248540,252208119996817,-0.38049316,1.202835100,-0.21313477


In [149]:
df_phone.shape

(3608408, 10)

In [150]:
df_phone.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3608408 entries, 0 to 3608407
Data columns (total 10 columns):
 #   Column      Dtype  
---  ------      -----  
 0   pessoa      int64  
 1   ativ        int64  
 2   tempo_acel  int64  
 3   acel_x_     float64
 4   acel_Y      float64
 5   acel_Z      float64
 6   temp_gyro   int64  
 7   giro_x      float64
 8   giro_Y_     float64
 9   giro_Z      float64
dtypes: float64(6), int64(4)
memory usage: 275.3 MB


In [151]:
df_phone.reset_index(drop = True,inplace=True)

In [152]:
df_phone.ativ.value_counts()

ativ
0     203919
12    202623
6     202620
7     202407
10    202395
3     202369
4     202350
16    202329
14    202329
17    202320
1     200251
13    198755
11    197914
8     197905
15    197894
2     197855
9     197841
5     194332
Name: count, dtype: int64

In [153]:
df_phone.head()

,pessoa,ativ,tempo_acel,acel_x_,acel_Y,acel_Z,temp_gyro,giro_x,giro_Y_,giro_Z
0,1600,0,252207666810782,-0.36476135,8.793503,1.05508420,252207918580802,-0.85321045,0.297225950,0.89018250
1,1600,0,252207717164786,-0.87973020,9.768784,1.01699830,252207968934806,-0.87513730,0.015472412,0.16223145
2,1600,0,252207767518790,2.00149540,11.109070,2.61915600,252208019288809,-0.72016907,0.388488770,-0.28401184
3,1600,0,252207817872794,0.45062256,12.651642,0.18455505,252208069642813,-0.57164000,1.227401700,-0.24166870
4,1600,0,252207868226798,-2.16435240,13.928436,-4.42248540,252208119996817,-0.38049316,1.202835100,-0.21313477


In [154]:
df_watch.reset_index(drop = True,inplace=True)

In [155]:
df_watch.shape

(3420018, 10)

In [156]:
df_watch.head()

,pessoa_x_watch,atividade_watch,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,0,90426708196641,7.0916250,-0.59166710,8.1955020,90426757696641,0.314944100,-1.02227650,-0.309961600
1,1600,0,90426757696641,4.9727570,-0.15831658,6.6967316,90426807196641,0.387382100,-0.61854124,-0.048971802
2,1600,0,90426807196641,3.2537200,-0.19183542,6.1077580,90426856696641,0.070998540,-0.20947972,-0.195978300
3,1600,0,90426856696641,2.8012160,-0.15592238,5.9976250,90426906196641,0.037975345,0.25497556,-0.156563500
4,1600,0,90426906196641,3.7708676,-1.05135380,7.7310270,90426955696641,0.073129070,0.71943086,-0.001034904


In [157]:
result_concatenation = custom_concat(df_watch.copy(),df_phone.copy())

In [158]:
result_concatenation.shape

(3231373, 20)

In [159]:
result_concatenation.head()

,pessoa_x_watch,atividade_watch,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z,pessoa,ativ,tempo_acel,acel_x_,acel_Y,acel_Z,temp_gyro,giro_x,giro_Y_,giro_Z
0,1600,0,90426708196641,7.0916250,-0.59166710,8.1955020,90426757696641,0.314944100,-1.02227650,-0.309961600,1600,0,252207666810782,-0.36476135,8.793503,1.05508420,252207918580802,-0.85321045,0.297225950,0.89018250
1,1600,0,90426757696641,4.9727570,-0.15831658,6.6967316,90426807196641,0.387382100,-0.61854124,-0.048971802,1600,0,252207717164786,-0.87973020,9.768784,1.01699830,252207968934806,-0.87513730,0.015472412,0.16223145
2,1600,0,90426807196641,3.2537200,-0.19183542,6.1077580,90426856696641,0.070998540,-0.20947972,-0.195978300,1600,0,252207767518790,2.00149540,11.109070,2.61915600,252208019288809,-0.72016907,0.388488770,-0.28401184
3,1600,0,90426856696641,2.8012160,-0.15592238,5.9976250,90426906196641,0.037975345,0.25497556,-0.156563500,1600,0,252207817872794,0.45062256,12.651642,0.18455505,252208069642813,-0.57164000,1.227401700,-0.24166870
4,1600,0,90426906196641,3.7708676,-1.05135380,7.7310270,90426955696641,0.073129070,0.71943086,-0.001034904,1600,0,252207868226798,-2.16435240,13.928436,-4.42248540,252208119996817,-0.38049316,1.202835100,-0.21313477


In [160]:
result_concatenation = result_concatenation.reset_index(drop=True)

In [161]:
result_concatenation.atividade_watch.value_counts().sort_index()

atividade_watch
0     180867
1     173496
2     174572
3     180932
4     176419
5     180051
6     182720
7     185286
8     176429
9     177335
10    180937
11    179122
12    180587
13    173691
14    183553
15    180011
16    182685
17    182680
Name: count, dtype: int64

In [162]:
result_concatenation.tail()


,pessoa_x_watch,atividade_watch,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z,pessoa,ativ,tempo_acel,acel_x_,acel_Y,acel_Z,temp_gyro,giro_x,giro_Y_,giro_Z
3231368,1650,17,2426184321449912,4.7689505,-5.4014688,7.5842323,2426184371437252,-2.8560646,2.0571015,0.85393304,1650,17,357626608042000,-7.2760696,-5.3025737,-1.8297950,357699155079000,0.041545313,-0.165116000,-0.177899170
3231369,1650,17,2426184371437252,2.1377227,-7.2114854,8.0654670,2426184421424592,-2.9114583,1.8174170,1.28110410,1650,17,357626628138000,-7.1443434,-5.4606447,-2.0190015,357699195384000,0.214118150,-0.291882460,-0.116113830
3231370,1650,17,2426184421424592,1.4865000,-13.3023340,5.3552310,2426184471411932,-3.0158541,1.5500356,2.05768180,1650,17,357626648248000,-7.1635036,-5.5348900,-2.1842580,357699235690000,0.517718550,-0.118244360,-0.103330650
3231371,1650,17,2426184471411932,4.0171714,-13.1467110,5.1182046,2426184521399272,-3.2182543,1.5958420,3.01855040,1650,17,357626668344000,-7.3143900,-5.6259010,-2.0980372,357699275996000,0.394147840,0.034088463,-0.071372720
3231372,1650,17,2426184521399272,4.4122143,-15.3182530,3.3704374,2426184571385262,-2.4555252,0.8661359,3.35410860,1650,17,357626688550000,-7.2497244,-5.9324636,-1.5926881,357699316371000,0.246076090,0.169377040,0.053263225


In [163]:
result_concatenation.shape

(3231373, 20)

In [164]:
test_diff_atividade = result_concatenation[result_concatenation['atividade_watch']!=result_concatenation['ativ']]

In [165]:
print(test_diff_atividade.shape[0])

0


In [166]:
test_diff_pessoa = result_concatenation[result_concatenation['pessoa_x_watch']!=result_concatenation['pessoa']]

In [167]:
print(len(test_diff_pessoa))

0


In [168]:
result_concatenation.columns

Index(['pessoa_x_watch', 'atividade_watch', 'tempo_acelerômetro',
       'acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'tempo_gyro',
       'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z', 'pessoa', 'ativ',
       'tempo_acel', 'acel_x_', 'acel_Y', 'acel_Z', 'temp_gyro', 'giro_x',
       'giro_Y_', 'giro_Z'],
      dtype='object')

In [169]:
result_concatenation = result_concatenation[['pessoa_x_watch', 'atividade_watch', 'tempo_acelerômetro',
       'acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'tempo_gyro',
       'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z','tempo_acel', 'acel_x_', 'acel_Y', 'acel_Z', 'temp_gyro', 
       'giro_x','giro_Y_', 'giro_Z']]


In [170]:
result_concatenation.head()

,pessoa_x_watch,atividade_watch,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z,tempo_acel,acel_x_,acel_Y,acel_Z,temp_gyro,giro_x,giro_Y_,giro_Z
0,1600,0,90426708196641,7.0916250,-0.59166710,8.1955020,90426757696641,0.314944100,-1.02227650,-0.309961600,252207666810782,-0.36476135,8.793503,1.05508420,252207918580802,-0.85321045,0.297225950,0.89018250
1,1600,0,90426757696641,4.9727570,-0.15831658,6.6967316,90426807196641,0.387382100,-0.61854124,-0.048971802,252207717164786,-0.87973020,9.768784,1.01699830,252207968934806,-0.87513730,0.015472412,0.16223145
2,1600,0,90426807196641,3.2537200,-0.19183542,6.1077580,90426856696641,0.070998540,-0.20947972,-0.195978300,252207767518790,2.00149540,11.109070,2.61915600,252208019288809,-0.72016907,0.388488770,-0.28401184
3,1600,0,90426856696641,2.8012160,-0.15592238,5.9976250,90426906196641,0.037975345,0.25497556,-0.156563500,252207817872794,0.45062256,12.651642,0.18455505,252208069642813,-0.57164000,1.227401700,-0.24166870
4,1600,0,90426906196641,3.7708676,-1.05135380,7.7310270,90426955696641,0.073129070,0.71943086,-0.001034904,252207868226798,-2.16435240,13.928436,-4.42248540,252208119996817,-0.38049316,1.202835100,-0.21313477


In [171]:
result_concatenation.head().columns

Index(['pessoa_x_watch', 'atividade_watch', 'tempo_acelerômetro',
       'acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'tempo_gyro',
       'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z', 'tempo_acel', 'acel_x_',
       'acel_Y', 'acel_Z', 'temp_gyro', 'giro_x', 'giro_Y_', 'giro_Z'],
      dtype='object')

In [172]:
result_concatenation.columns = ['pessoa_x', 'atividade', 'tempo_acelerômetro_watch',
       'acelerômetro_x_watch', 'acelerômetro_Y_watch', 'acelerômetro_Z_watch', 'tempo_gyro_watch',
       'giroscópio_x_watch', 'giroscópio_Y_watch', 'giroscópio_Z_watch', 'tempo_acelerômetro_phone', 'acelerômetro_x_phone',
       'acelerômetro_Y_phone', 'acelerômetro_Z_phone', 'tempo_giroscópio_phone',
         'giroscópio_x_phone', 'giroscópio_Y_phone', 'giroscópio_Z_phone']

In [173]:
result_concatenation.head()

,pessoa_x,atividade,tempo_acelerômetro_watch,acelerômetro_x_watch,acelerômetro_Y_watch,acelerômetro_Z_watch,tempo_gyro_watch,giroscópio_x_watch,giroscópio_Y_watch,giroscópio_Z_watch,tempo_acelerômetro_phone,acelerômetro_x_phone,acelerômetro_Y_phone,acelerômetro_Z_phone,tempo_giroscópio_phone,giroscópio_x_phone,giroscópio_Y_phone,giroscópio_Z_phone
0,1600,0,90426708196641,7.0916250,-0.59166710,8.1955020,90426757696641,0.314944100,-1.02227650,-0.309961600,252207666810782,-0.36476135,8.793503,1.05508420,252207918580802,-0.85321045,0.297225950,0.89018250
1,1600,0,90426757696641,4.9727570,-0.15831658,6.6967316,90426807196641,0.387382100,-0.61854124,-0.048971802,252207717164786,-0.87973020,9.768784,1.01699830,252207968934806,-0.87513730,0.015472412,0.16223145
2,1600,0,90426807196641,3.2537200,-0.19183542,6.1077580,90426856696641,0.070998540,-0.20947972,-0.195978300,252207767518790,2.00149540,11.109070,2.61915600,252208019288809,-0.72016907,0.388488770,-0.28401184
3,1600,0,90426856696641,2.8012160,-0.15592238,5.9976250,90426906196641,0.037975345,0.25497556,-0.156563500,252207817872794,0.45062256,12.651642,0.18455505,252208069642813,-0.57164000,1.227401700,-0.24166870
4,1600,0,90426906196641,3.7708676,-1.05135380,7.7310270,90426955696641,0.073129070,0.71943086,-0.001034904,252207868226798,-2.16435240,13.928436,-4.42248540,252208119996817,-0.38049316,1.202835100,-0.21313477


In [174]:
result_concatenation.head()

,pessoa_x,atividade,tempo_acelerômetro_watch,acelerômetro_x_watch,acelerômetro_Y_watch,acelerômetro_Z_watch,tempo_gyro_watch,giroscópio_x_watch,giroscópio_Y_watch,giroscópio_Z_watch,tempo_acelerômetro_phone,acelerômetro_x_phone,acelerômetro_Y_phone,acelerômetro_Z_phone,tempo_giroscópio_phone,giroscópio_x_phone,giroscópio_Y_phone,giroscópio_Z_phone
0,1600,0,90426708196641,7.0916250,-0.59166710,8.1955020,90426757696641,0.314944100,-1.02227650,-0.309961600,252207666810782,-0.36476135,8.793503,1.05508420,252207918580802,-0.85321045,0.297225950,0.89018250
1,1600,0,90426757696641,4.9727570,-0.15831658,6.6967316,90426807196641,0.387382100,-0.61854124,-0.048971802,252207717164786,-0.87973020,9.768784,1.01699830,252207968934806,-0.87513730,0.015472412,0.16223145
2,1600,0,90426807196641,3.2537200,-0.19183542,6.1077580,90426856696641,0.070998540,-0.20947972,-0.195978300,252207767518790,2.00149540,11.109070,2.61915600,252208019288809,-0.72016907,0.388488770,-0.28401184
3,1600,0,90426856696641,2.8012160,-0.15592238,5.9976250,90426906196641,0.037975345,0.25497556,-0.156563500,252207817872794,0.45062256,12.651642,0.18455505,252208069642813,-0.57164000,1.227401700,-0.24166870
4,1600,0,90426906196641,3.7708676,-1.05135380,7.7310270,90426955696641,0.073129070,0.71943086,-0.001034904,252207868226798,-2.16435240,13.928436,-4.42248540,252208119996817,-0.38049316,1.202835100,-0.21313477


In [175]:
# Define the mapping dictionary
mapping_dict = {'A':0,'B':1,'C':2,'D':3,'E':4,'F':5,'G':6,'H':7,'I':8,'J':9,'K':10,'L':11,'M':12,'O':13,'P':14,'Q':15,'R':16,'S':17}

In [176]:
result_concatenation["atividade"] = result_concatenation["atividade"].replace(to_replace=mapping_dict)

In [177]:
result_concatenation.head()

,pessoa_x,atividade,tempo_acelerômetro_watch,acelerômetro_x_watch,acelerômetro_Y_watch,acelerômetro_Z_watch,tempo_gyro_watch,giroscópio_x_watch,giroscópio_Y_watch,giroscópio_Z_watch,tempo_acelerômetro_phone,acelerômetro_x_phone,acelerômetro_Y_phone,acelerômetro_Z_phone,tempo_giroscópio_phone,giroscópio_x_phone,giroscópio_Y_phone,giroscópio_Z_phone
0,1600,0,90426708196641,7.0916250,-0.59166710,8.1955020,90426757696641,0.314944100,-1.02227650,-0.309961600,252207666810782,-0.36476135,8.793503,1.05508420,252207918580802,-0.85321045,0.297225950,0.89018250
1,1600,0,90426757696641,4.9727570,-0.15831658,6.6967316,90426807196641,0.387382100,-0.61854124,-0.048971802,252207717164786,-0.87973020,9.768784,1.01699830,252207968934806,-0.87513730,0.015472412,0.16223145
2,1600,0,90426807196641,3.2537200,-0.19183542,6.1077580,90426856696641,0.070998540,-0.20947972,-0.195978300,252207767518790,2.00149540,11.109070,2.61915600,252208019288809,-0.72016907,0.388488770,-0.28401184
3,1600,0,90426856696641,2.8012160,-0.15592238,5.9976250,90426906196641,0.037975345,0.25497556,-0.156563500,252207817872794,0.45062256,12.651642,0.18455505,252208069642813,-0.57164000,1.227401700,-0.24166870
4,1600,0,90426906196641,3.7708676,-1.05135380,7.7310270,90426955696641,0.073129070,0.71943086,-0.001034904,252207868226798,-2.16435240,13.928436,-4.42248540,252208119996817,-0.38049316,1.202835100,-0.21313477


In [178]:
result_concatenation.shape

(3231373, 18)

In [179]:
result_concatenation.tail(50)

,pessoa_x,atividade,tempo_acelerômetro_watch,acelerômetro_x_watch,acelerômetro_Y_watch,acelerômetro_Z_watch,tempo_gyro_watch,giroscópio_x_watch,giroscópio_Y_watch,giroscópio_Z_watch,tempo_acelerômetro_phone,acelerômetro_x_phone,acelerômetro_Y_phone,acelerômetro_Z_phone,tempo_giroscópio_phone,giroscópio_x_phone,giroscópio_Y_phone,giroscópio_Z_phone
3231323,1650,17,2426182071909922,2.68120670,-4.80291840,8.1588410,2426182121897632,-1.969764500,1.562818800,-0.003604855,357625701021000,-6.6198344,-6.3372220,-2.0309765,357697341426000,-0.220509740,0.009587380,-0.059654810
3231324,1650,17,2426182121897632,0.72035545,-4.22591540,12.9472450,2426182171885342,1.064108600,1.739652800,0.359650300,357625721220000,-6.9168167,-5.9324636,-2.1459377,357697381707000,-0.191747600,0.181094960,-0.054328486
3231325,1650,17,2426182171885342,0.26785132,-10.12044050,10.8307700,2426182221873052,4.321687000,2.470424200,-0.798292100,357625741316000,-7.1970340,-5.5444703,-2.2249732,357697422007000,-0.006391587,0.147006500,0.029827405
3231326,1650,17,2426182221873052,-1.84144040,3.92155270,7.8021050,2426182271860762,1.949343300,2.141257300,-1.457690800,357625761598000,-7.5467067,-5.2690430,-2.2273684,357697462212000,0.149137020,0.019174760,0.084155895
3231327,1650,17,2426182271860762,-3.95552040,6.04999830,6.0016650,2426182321848472,1.746943100,2.089059400,-1.445972900,357625781694000,-7.7311234,-4.8930250,-2.1076174,357697502480000,0.155528600,-0.095873800,0.098004326
3231328,1650,17,2426182321848472,-8.68406900,-0.68249714,5.1684830,2426182371836182,1.905667500,1.122864600,-0.977256540,357625801924000,-7.7957890,-4.4667115,-2.0142114,357697542846000,0.005326322,-0.089482220,0.062850600
3231329,1650,17,2426182371836182,-7.83652160,0.83303255,4.8404775,2426182421823892,-0.828866240,0.741499960,0.061376274,357625822020000,-7.7814190,-4.1194340,-1.9184108,357697583164000,-0.172572840,-0.043675844,0.008522116
3231330,1650,17,2426182421823892,-9.03601600,-0.97219550,4.0671500,2426182471816332,0.509105860,0.121516040,-0.069651246,357625842216000,-7.8317140,-4.0403980,-1.7172291,357697623461000,-0.216248690,-0.079894830,-0.012783173
3231331,1650,17,2426182471816332,-8.24353600,-0.31857845,6.3033347,2426182521808772,0.791400900,-0.638017500,-0.512801230,357625862312000,-7.7311234,-4.2870850,-1.4394068,357697663616000,-0.133158060,-0.117179090,-0.027696876
3231332,1650,17,2426182521808772,-6.04565860,2.34616800,6.8923080,2426182571801212,0.905384240,-1.036426400,-0.518127560,357625882480000,-7.6616680,-4.6295730,-1.3244458,357697703923000,-0.035153728,-0.116113830,-0.041545313


In [180]:
result_concatenation.to_csv(WORKDIR + 'Data/processed/activity_recognition_data_WATCH_PHONE_concat_by_comparacao_grupos.csv', index=False)